In [1]:
# import tools for selecting the correct python executable
import os
import sys

# use the same python executable for the driver and workers
python_path = sys.executable

os.environ["PYSPARK_PYTHON"] = python_path
os.environ["PYSPARK_DRIVER_PYTHON"] = python_path

# import sparksession after setting the python paths
from pyspark.sql import SparkSession

# create a new spark session using the same python version
spark = (
    SparkSession.builder
    .appName("vehicle_location_preprocessing")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.session.timeZone", "UTC")
    .config(
        "spark.pyspark.python",
        python_path
    )
    .config(
        "spark.pyspark.driver.python",
        python_path
    )
    .config(
        "spark.executorEnv.PYSPARK_PYTHON",
        python_path
    )
    .getOrCreate()
)

# reduce unnecessary spark messages
spark.sparkContext.setLogLevel("WARN")

print("driver python version:", sys.version.split()[0])
print("driver python path:", python_path)
print(
    "spark worker python:",
    spark.sparkContext.pythonExec
)
print("pyspark version:", spark.version)
print(
    "shuffle partitions:",
    spark.conf.get("spark.sql.shuffle.partitions")
)
print("spark ui:", spark.sparkContext.uiWebUrl)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/12 22:02:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/07/12 22:02:41 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


driver python version: 3.11.15
driver python path: /opt/anaconda3/envs/pyspark35/bin/python
spark worker python: /opt/anaconda3/envs/pyspark35/bin/python
pyspark version: 3.5.1
shuffle partitions: 4
spark ui: http://192.168.1.13:4041


In [2]:

from pathlib import Path
# import elementtree for reading xml files
import xml.etree.ElementTree as ET
# import datetime tools for snapshot timestamps
from datetime import datetime, timezone
# import regular expressions for reading timestamps from filenames
import re
# import pyspark functions for cleaning and analysis
from pyspark.sql import functions as F
# import pyspark data types for creating a fixed schema
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType
)
print("libraries imported successfully")

libraries imported successfully


In [3]:
# folder where the notebook is running
current_folder = Path.cwd()
if current_folder.name == "notebooks":
    project_root = current_folder.parent
else:
    project_root = current_folder
vehicle_folder = (
    project_root
    / "data"
    / "raw"
    / "vehicle_location"
)
processed_folder = (
    project_root
    / "data"
    / "processed"
    / "vehicle_locations"
)
print("current folder:", current_folder)
print("project root:", project_root)
print("vehicle folder:", vehicle_folder)
print("vehicle folder exists:", vehicle_folder.exists())
print("processed folder:", processed_folder)

current folder: /Users/babitaadhikari/Desktop/bus-disruption-platform/notebooks
project root: /Users/babitaadhikari/Desktop/bus-disruption-platform
vehicle folder: /Users/babitaadhikari/Desktop/bus-disruption-platform/data/raw/vehicle_location
vehicle folder exists: True
processed folder: /Users/babitaadhikari/Desktop/bus-disruption-platform/data/processed/vehicle_locations


In [4]:
# find every xml file inside the vehicle location folder
xml_files = sorted(
    vehicle_folder.glob("*.xml")
)
print("number of xml files found:", len(xml_files))
# display every xml filename
for xml_path in xml_files:
    print(xml_path.name)
# stop the notebook when no xml files are found
if not xml_files:
    raise FileNotFoundError(
        f"no xml files were found in {vehicle_folder}"
    )

number of xml files found: 2
nx_west_midlands_20260711T174953Z.xml
nx_west_midlands_20260712T041703Z.xml


In [5]:
#  function that finds text inside an xml tag
def get_xml_text(element, tag_name):
    found_element = element.find(
        f".//{{*}}{tag_name}"
    )
    if found_element is None:
        return None
    if found_element.text is None:
        return None
    cleaned_text = found_element.text.strip()

    if cleaned_text == "":
        return None
    return cleaned_text
print("xml text function created")

xml text function created


In [6]:
#  function that extracts the timestamp from an xml filename
def timestamp_from_filename(filename):
    pattern = r"(\d{8}T\d{6}Z)"
    match = re.search(
        pattern,
        filename
    )
    if match is None:
        return None
    timestamp_text = match.group(1)
    parsed_time = datetime.strptime(
        timestamp_text,
        "%Y%m%dT%H%M%SZ"
    )
    parsed_time = parsed_time.replace(
        tzinfo=timezone.utc
    )
    return parsed_time.isoformat()
print("filename timestamp function created")

filename timestamp function created


In [7]:
# test  timestamp function using the first xml filename
test_filename_time = timestamp_from_filename(
    xml_files[0].name
)
print("first filename:", xml_files[0].name)
print("timestamp from filename:", test_filename_time)

first filename: nx_west_midlands_20260711T174953Z.xml
timestamp from filename: 2026-07-11T17:49:53+00:00


## xml parser

In [8]:
#  function that converts one xml file into vehicle records
def parse_vehicle_xml(xml_path):
    records = []
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
    except ET.ParseError as error:
        print(
            "xml parsing failed for",
            xml_path.name,
            error
        )
        return records
    # obtain the response timestamp from the xml
    response_timestamp = get_xml_text(
        root,
        "ResponseTimestamp"
    )
    # use the filename timestamp when needed
    if response_timestamp is None:
        response_timestamp = timestamp_from_filename(
            xml_path.name
        )
    # find every vehicle activity inside the xml
    vehicle_activities = root.findall(
        ".//{*}VehicleActivity"
    )
    # convert every vehicle activity into one dictionary
    for activity in vehicle_activities:
        record = {
            "source_file": xml_path.name,
            "snapshot_time_raw": response_timestamp,

            "recorded_at_time_raw": get_xml_text(
                activity,
                "RecordedAtTime"
            ),

            "valid_until_time_raw": get_xml_text(
                activity,
                "ValidUntilTime"
            ),

            "operator_ref": get_xml_text(
                activity,
                "OperatorRef"
            ),

            "line_ref": get_xml_text(
                activity,
                "LineRef"
            ),

            "published_line_name": get_xml_text(
                activity,
                "PublishedLineName"
            ),

            "direction_ref": get_xml_text(
                activity,
                "DirectionRef"
            ),

            "dated_vehicle_journey_ref": get_xml_text(
                activity,
                "DatedVehicleJourneyRef"
            ),

            "journey_pattern_ref": get_xml_text(
                activity,
                "JourneyPatternRef"
            ),

            "vehicle_ref": get_xml_text(
                activity,
                "VehicleRef"
            ),

            "block_ref": get_xml_text(
                activity,
                "BlockRef"
            ),

            "origin_ref": get_xml_text(
                activity,
                "OriginRef"
            ),

            "destination_ref": get_xml_text(
                activity,
                "DestinationRef"
            ),

            "destination_name": get_xml_text(
                activity,
                "DestinationName"
            ),

            "origin_aimed_departure_time_raw": get_xml_text(
                activity,
                "OriginAimedDepartureTime"
            ),

            "longitude_raw": get_xml_text(
                activity,
                "Longitude"
            ),

            "latitude_raw": get_xml_text(
                activity,
                "Latitude"
            ),

            "bearing_raw": get_xml_text(
                activity,
                "Bearing"
            )
        }

        records.append(record)
    return records
print("vehicle xml parser created")

vehicle xml parser created


In [9]:
# parse only the first xml file for testing
test_records = parse_vehicle_xml(
    xml_files[0]
)
print("test file:", xml_files[0].name)
print(
    "vehicle activity records:",
    len(test_records)
)
# display the first extracted vehicle record
if test_records:
    for key, value in test_records[0].items():
        print(f"{key}: {value}")

test file: nx_west_midlands_20260711T174953Z.xml
vehicle activity records: 1146
source_file: nx_west_midlands_20260711T174953Z.xml
snapshot_time_raw: 2026-07-11T17:48:58.184+00:00
recorded_at_time_raw: 2026-07-10T17:50:13+00:00
valid_until_time_raw: 2026-07-11T17:53:58.184+00:00
operator_ref: TCVW
line_ref: 17
published_line_name: 17
direction_ref: inbound
dated_vehicle_journey_ref: 104
journey_pattern_ref: None
vehicle_ref: 2111
block_ref: CV906
origin_ref: 43000038206
destination_ref: 43000127902
destination_name: Finham Fenside Avenue
origin_aimed_departure_time_raw: 2026-07-10T16:52:00+00:00
longitude_raw: -1.492329
latitude_raw: 52.388493
bearing_raw: 12


In [10]:
# create a list for observations from every xml file
all_vehicle_records = []
# create a list for file level record counts
file_record_counts = []
# process every xml snapshot
for xml_path in xml_files:
    file_records = parse_vehicle_xml(
        xml_path
    )
    all_vehicle_records.extend(
        file_records
    )
    file_record_counts.append(
        {
            "source_file": xml_path.name,
            "record_count": len(file_records)
        }
    )
    print(
        xml_path.name,
        ":",
        f"{len(file_records):,}",
        "records"
    )
print("-" * 50)
print(
    "total parsed vehicle observations:",
    f"{len(all_vehicle_records):,}"
)

nx_west_midlands_20260711T174953Z.xml : 1,146 records
nx_west_midlands_20260712T041703Z.xml : 1,051 records
--------------------------------------------------
total parsed vehicle observations: 2,197


In [11]:
# stop the notebook when xml parsing produces no records
if not all_vehicle_records:
    raise ValueError(
        "no vehicle activity records were extracted"
    )
print("xml parsing completed successfully")

xml parsing completed successfully


In [12]:
# create  fixed schema for the vehicle location dataframe
vehicle_schema = StructType([
    StructField("source_file", StringType(), True),
    StructField("snapshot_time_raw", StringType(), True),
    StructField("recorded_at_time_raw", StringType(), True),
    StructField("valid_until_time_raw", StringType(), True),
    StructField("operator_ref", StringType(), True),
    StructField("line_ref", StringType(), True),
    StructField("published_line_name", StringType(), True),
    StructField("direction_ref", StringType(), True),
    StructField(
        "dated_vehicle_journey_ref",
        StringType(),
        True
    ),
    StructField(
        "journey_pattern_ref",
        StringType(),
        True
    ),
    StructField("vehicle_ref", StringType(), True),
    StructField("block_ref", StringType(), True),
    StructField("origin_ref", StringType(), True),
    StructField("destination_ref", StringType(), True),
    StructField("destination_name", StringType(), True),
    StructField(
        "origin_aimed_departure_time_raw",
        StringType(),
        True
    ),
    StructField("longitude_raw", StringType(), True),
    StructField("latitude_raw", StringType(), True),
    StructField("bearing_raw", StringType(), True)
])
print("vehicle schema created")

vehicle schema created


In [13]:
# convert the parsed python records into a pyspark dataframe
vehicle_raw_df = spark.createDataFrame(
    all_vehicle_records,
    schema=vehicle_schema
)
print(
    "raw vehicle observations:",
    f"{vehicle_raw_df.count():,}"
)
print(
    "initial partitions:",
    vehicle_raw_df.rdd.getNumPartitions()
)

raw vehicle observations: 2,197
initial partitions: 8


In [14]:
# display the structure of the raw dataframe
vehicle_raw_df.printSchema()
# display sample vehicle observations
vehicle_raw_df.select(
    "source_file",
    "recorded_at_time_raw",
    "operator_ref",
    "line_ref",
    "dated_vehicle_journey_ref",
    "vehicle_ref",
    "longitude_raw",
    "latitude_raw"
).show(
    10,
    truncate=False
)

root
 |-- source_file: string (nullable = true)
 |-- snapshot_time_raw: string (nullable = true)
 |-- recorded_at_time_raw: string (nullable = true)
 |-- valid_until_time_raw: string (nullable = true)
 |-- operator_ref: string (nullable = true)
 |-- line_ref: string (nullable = true)
 |-- published_line_name: string (nullable = true)
 |-- direction_ref: string (nullable = true)
 |-- dated_vehicle_journey_ref: string (nullable = true)
 |-- journey_pattern_ref: string (nullable = true)
 |-- vehicle_ref: string (nullable = true)
 |-- block_ref: string (nullable = true)
 |-- origin_ref: string (nullable = true)
 |-- destination_ref: string (nullable = true)
 |-- destination_name: string (nullable = true)
 |-- origin_aimed_departure_time_raw: string (nullable = true)
 |-- longitude_raw: string (nullable = true)
 |-- latitude_raw: string (nullable = true)
 |-- bearing_raw: string (nullable = true)

+-------------------------------------+-------------------------+------------+--------+-------

In [15]:
# create a function to remove spaces from string columns
def trim_string_columns(dataframe):
    cleaned_df = dataframe
    for column_name, data_type in dataframe.dtypes:
        if data_type == "string":
            cleaned_df = cleaned_df.withColumn(
                column_name,
                F.trim(F.col(column_name))
            )
    return cleaned_df
# clean all string columns
vehicle_clean_df = trim_string_columns(
    vehicle_raw_df
)
print("string columns cleaned")

string columns cleaned


In [16]:
# convert timestamp text into spark timestamp columns
# convert coordinate text into numeric columns
vehicle_clean_df = (
    vehicle_clean_df
    .withColumn(
        "snapshot_time",
        F.to_timestamp(
            F.col("snapshot_time_raw")
        )
    )
    .withColumn(
        "recorded_at_time",
        F.to_timestamp(
            F.col("recorded_at_time_raw")
        )
    )
    .withColumn(
        "valid_until_time",
        F.to_timestamp(
            F.col("valid_until_time_raw")
        )
    )
    .withColumn(
        "origin_aimed_departure_time",
        F.to_timestamp(
            F.col(
                "origin_aimed_departure_time_raw"
            )
        )
    )
    .withColumn(
        "longitude",
        F.col("longitude_raw").cast("double")
    )

    .withColumn(
        "latitude",
        F.col("latitude_raw").cast("double")
    )

    .withColumn(
        "bearing",
        F.col("bearing_raw").cast("double")
    )
)
print("timestamps and numeric values converted")

timestamps and numeric values converted


In [17]:
# compare the original values with the converted values
vehicle_clean_df.select(
    "recorded_at_time_raw",
    "recorded_at_time",
    "origin_aimed_departure_time_raw",
    "origin_aimed_departure_time",
    "longitude",
    "latitude",
    "bearing"
).show(
    10,
    truncate=False
)

+-------------------------+-------------------+-------------------------------+---------------------------+---------+---------+-------+
|recorded_at_time_raw     |recorded_at_time   |origin_aimed_departure_time_raw|origin_aimed_departure_time|longitude|latitude |bearing|
+-------------------------+-------------------+-------------------------------+---------------------------+---------+---------+-------+
|2026-07-10T17:50:13+00:00|2026-07-10 17:50:13|2026-07-10T16:52:00+00:00      |2026-07-10 16:52:00        |-1.492329|52.388493|12.0   |
|2026-07-10T18:15:46+00:00|2026-07-10 18:15:46|2026-07-10T17:20:00+00:00      |2026-07-10 17:20:00        |-1.467612|52.514935|212.0  |
|2026-07-11T17:43:18+00:00|2026-07-11 17:43:18|2026-07-11T17:17:00+00:00      |2026-07-11 17:17:00        |-1.504132|52.410824|0.0    |
|2026-07-11T17:13:20+00:00|2026-07-11 17:13:20|2026-07-11T16:47:00+00:00      |2026-07-11 16:47:00        |-1.504137|52.410931|0.0    |
|2026-07-10T18:09:20+00:00|2026-07-10 18:09:20|2

In [18]:
# define important columns for data quality checking
important_columns = [
    "source_file",
    "snapshot_time",
    "recorded_at_time",
    "operator_ref",
    "line_ref",
    "dated_vehicle_journey_ref",
    "vehicle_ref",
    "longitude",
    "latitude"
]
# create calculations for missing values
missing_expressions = []

for column_name in important_columns:
    missing_expressions.append(
        F.sum(
            F.when(
                F.col(column_name).isNull(),
                1
            ).otherwise(0)
        ).alias(column_name)
    )
print("missing values in important columns")
vehicle_clean_df.select(
    missing_expressions
).show(
    truncate=False
)

missing values in important columns


[Stage 5:>                                                          (0 + 8) / 8]

+-----------+-------------+----------------+------------+--------+-------------------------+-----------+---------+--------+
|source_file|snapshot_time|recorded_at_time|operator_ref|line_ref|dated_vehicle_journey_ref|vehicle_ref|longitude|latitude|
+-----------+-------------+----------------+------------+--------+-------------------------+-----------+---------+--------+
|0          |0            |0               |0           |0       |0                        |0          |0        |0       |
+-----------+-------------+----------------+------------+--------+-------------------------+-----------+---------+--------+



In [19]:
# define missing or impossible coordinate conditions
invalid_coordinate_condition = (
    F.col("latitude").isNull()
    | F.col("longitude").isNull()
    | (F.col("latitude") < -90)
    | (F.col("latitude") > 90)
    | (F.col("longitude") < -180)
    | (F.col("longitude") > 180)
)
# count records with invalid coordinates
invalid_coordinate_count = (
    vehicle_clean_df
    .filter(invalid_coordinate_condition)
    .count()
)
print(
    "invalid coordinate records:",
    f"{invalid_coordinate_count:,}"
)

invalid coordinate records: 0


In [20]:
# keep records that contain the required information
vehicle_clean_df = (
    vehicle_clean_df

    .filter(
        F.col("recorded_at_time").isNotNull()
    )

    .filter(
        F.col("operator_ref").isNotNull()
    )

    .filter(
        F.col("line_ref").isNotNull()
    )

    .filter(
        F.col("vehicle_ref").isNotNull()
    )

    .filter(
        ~invalid_coordinate_condition
    )
)

print(
    "rows after removing incomplete records:",
    f"{vehicle_clean_df.count():,}"
)

rows after removing incomplete records: 2,197


In [21]:
# keep records that contain the required information
vehicle_clean_df = (
    vehicle_clean_df

    .filter(
        F.col("recorded_at_time").isNotNull()
    )

    .filter(
        F.col("operator_ref").isNotNull()
    )

    .filter(
        F.col("line_ref").isNotNull()
    )

    .filter(
        F.col("vehicle_ref").isNotNull()
    )

    .filter(
        ~invalid_coordinate_condition
    )
)
print(
    "rows after removing incomplete records:",
    f"{vehicle_clean_df.count():,}"
)

[Stage 14:>                                                         (0 + 8) / 8]

rows after removing incomplete records: 2,197


In [22]:
# create useful time columns for later disruption analysis
vehicle_clean_df = (
    vehicle_clean_df

    .withColumn(
        "observation_date",
        F.to_date(
            F.col("recorded_at_time")
        )
    )

    .withColumn(
        "observation_hour",
        F.hour(
            F.col("recorded_at_time")
        )
    )

    .withColumn(
        "observation_day_of_week",
        F.dayofweek(
            F.col("recorded_at_time")
        )
    )

    .withColumn(
        "observation_age_seconds",
        F.col("snapshot_time").cast("long")
        - F.col("recorded_at_time").cast("long")
    )
)
print("time based features created")

time based features created


In [23]:
# display the new time based columns
vehicle_clean_df.select(
    "recorded_at_time",
    "snapshot_time",
    "observation_date",
    "observation_hour",
    "observation_day_of_week",
    "observation_age_seconds"
).show(
    10,
    truncate=False
)

+-------------------+-----------------------+----------------+----------------+-----------------------+-----------------------+
|recorded_at_time   |snapshot_time          |observation_date|observation_hour|observation_day_of_week|observation_age_seconds|
+-------------------+-----------------------+----------------+----------------+-----------------------+-----------------------+
|2026-07-10 17:50:13|2026-07-11 17:48:58.184|2026-07-10      |17              |6                      |86325                  |
|2026-07-10 18:15:46|2026-07-11 17:48:58.184|2026-07-10      |18              |6                      |84792                  |
|2026-07-11 17:43:18|2026-07-11 17:48:58.184|2026-07-11      |17              |7                      |340                    |
|2026-07-11 17:13:20|2026-07-11 17:48:58.184|2026-07-11      |17              |7                      |2138                   |
|2026-07-10 18:09:20|2026-07-11 17:48:58.184|2026-07-10      |18              |6                      |8

In [24]:
# count observations with negative age values
negative_age_count = (
    vehicle_clean_df
    .filter(
        F.col("observation_age_seconds") < 0
    )
    .count()
)
# count observations older than fifteen minutes
old_observation_count = (
    vehicle_clean_df
    .filter(
        F.col("observation_age_seconds") > 900
    )
    .count()
)

print(
    "negative observation ages:",
    f"{negative_age_count:,}"
)

print(
    "observations older than fifteen minutes:",
    f"{old_observation_count:,}"
)

negative observation ages: 0
observations older than fifteen minutes: 1,459


In [25]:
# define the columns needed for later joining and analysis
final_columns = [
    "source_file",
    "snapshot_time",
    "recorded_at_time",
    "valid_until_time",
    "operator_ref",
    "line_ref",
    "published_line_name",
    "direction_ref",
    "dated_vehicle_journey_ref",
    "journey_pattern_ref",
    "vehicle_ref",
    "block_ref",
    "origin_ref",
    "destination_ref",
    "destination_name",
    "origin_aimed_departure_time",
    "longitude",
    "latitude",
    "bearing",
    "observation_date",
    "observation_hour",
    "observation_day_of_week",
    "observation_age_seconds"
]
# keep only columns that exist in the dataframe
available_final_columns = [
    column_name
    for column_name in final_columns
    if column_name in vehicle_clean_df.columns
]
# select the final columns
vehicle_clean_df = vehicle_clean_df.select(
    *available_final_columns
)
print("final vehicle location columns")
for column_name in vehicle_clean_df.columns:
    print(column_name)

final vehicle location columns
source_file
snapshot_time
recorded_at_time
valid_until_time
operator_ref
line_ref
published_line_name
direction_ref
dated_vehicle_journey_ref
journey_pattern_ref
vehicle_ref
block_ref
origin_ref
destination_ref
destination_name
origin_aimed_departure_time
longitude
latitude
bearing
observation_date
observation_hour
observation_day_of_week
observation_age_seconds


In [26]:
# distribute records across four partitions using the bus line
vehicle_clean_df = vehicle_clean_df.repartition(
    4,
    "line_ref"
)
print(
    "partitions after repartition:",
    vehicle_clean_df.rdd.getNumPartitions()
)

[Stage 24:>                                                         (0 + 8) / 8]

partitions after repartition: 4


In [27]:
# cache the processed dataframe because it will be reused
vehicle_clean_df.cache()
# trigger spark processing and store the final row count
processed_vehicle_count = (
    vehicle_clean_df.count()
)
print(
    "processed vehicle observations:",
    f"{processed_vehicle_count:,}"
)
print(
    "cached partitions:",
    vehicle_clean_df.rdd.getNumPartitions()
)

processed vehicle observations: 2,197
cached partitions: 4


In [28]:
# display sample cleaned vehicle observations
vehicle_clean_df.show(
    10,
    truncate=False
)

+-------------------------------------+-----------------------+-------------------+-----------------------+------------+--------+-------------------+-------------+-------------------------+-------------------+-----------+---------+-----------+---------------+------------------------------------+---------------------------+---------+---------+-------+----------------+----------------+-----------------------+-----------------------+
|source_file                          |snapshot_time          |recorded_at_time   |valid_until_time       |operator_ref|line_ref|published_line_name|direction_ref|dated_vehicle_journey_ref|journey_pattern_ref|vehicle_ref|block_ref|origin_ref |destination_ref|destination_name                    |origin_aimed_departure_time|longitude|latitude |bearing|observation_date|observation_hour|observation_day_of_week|observation_age_seconds|
+-------------------------------------+-----------------------+-------------------+-----------------------+------------+--------+-

In [29]:
# calculate the main summary values
vehicle_summary = vehicle_clean_df.agg(
    F.count("*").alias(
        "total_observations"
    ),

    F.countDistinct(
        "source_file"
    ).alias(
        "xml_files"
    ),

    F.countDistinct(
        "vehicle_ref"
    ).alias(
        "distinct_vehicles"
    ),

    F.countDistinct(
        "line_ref"
    ).alias(
        "distinct_lines"
    ),

    F.countDistinct(
        "operator_ref"
    ).alias(
        "distinct_operators"
    ),

    F.min(
        "recorded_at_time"
    ).alias(
        "earliest_observation"
    ),

    F.max(
        "recorded_at_time"
    ).alias(
        "latest_observation"
    )
)

vehicle_summary.show(
    truncate=False
)

[Stage 36:>                                                         (0 + 4) / 4]

+------------------+---------+-----------------+--------------+------------------+--------------------+-------------------+
|total_observations|xml_files|distinct_vehicles|distinct_lines|distinct_operators|earliest_observation|latest_observation |
+------------------+---------+-----------------+--------------+------------------+--------------------+-------------------+
|2197              |2        |1147             |111           |2                 |2026-07-10 17:50:13 |2026-07-12 04:16:45|
+------------------+---------+-----------------+--------------+------------------+--------------------+-------------------+



In [30]:
# count how many cleaned records came from each xml file
observations_by_file = (
    vehicle_clean_df
    .groupBy("source_file")
    .agg(
        F.count("*").alias(
            "vehicle_observations"
        )
    )
    .orderBy("source_file")
)
observations_by_file.show(
    truncate=False
)

+-------------------------------------+--------------------+
|source_file                          |vehicle_observations|
+-------------------------------------+--------------------+
|nx_west_midlands_20260711T174953Z.xml|1146                |
|nx_west_midlands_20260712T041703Z.xml|1051                |
+-------------------------------------+--------------------+



In [31]:
# calculate observations and vehicles for each bus line
busiest_lines = (
    vehicle_clean_df
    .groupBy(
        "line_ref",
        "published_line_name"
    )
    .agg(
        F.count("*").alias(
            "observation_count"
        ),

        F.countDistinct(
            "vehicle_ref"
        ).alias(
            "distinct_vehicles"
        )
    )
    .orderBy(
        F.desc("observation_count")
    )
)

busiest_lines.show(
    10,
    truncate=False
)

+--------+-------------------+-----------------+-----------------+
|line_ref|published_line_name|observation_count|distinct_vehicles|
+--------+-------------------+-----------------+-----------------+
|6       |6                  |80               |47               |
|74      |74                 |55               |29               |
|4       |4                  |54               |34               |
|9       |9                  |52               |29               |
|7       |7                  |52               |31               |
|11A     |11A                |48               |34               |
|3       |3                  |47               |28               |
|16      |16                 |42               |24               |
|11C     |11C                |41               |30               |
|11      |11                 |41               |25               |
+--------+-------------------+-----------------+-----------------+
only showing top 10 rows



In [32]:
# register the dataframe as a temporary sql table
vehicle_clean_df.createOrReplaceTempView(
    "vehicle_locations"
)
print("temporary sql view created")
print("view name: vehicle_locations")

temporary sql view created
view name: vehicle_locations


26/07/12 22:11:40 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [33]:
# count observations and vehicles for each bus line
line_sql_summary = spark.sql("""
    SELECT
        line_ref,
        published_line_name,
        COUNT(*) AS observation_count,
        COUNT(DISTINCT vehicle_ref) AS vehicle_count
    FROM vehicle_locations
    GROUP BY
        line_ref,
        published_line_name
    ORDER BY observation_count DESC
    LIMIT 10
""")
line_sql_summary.show(
    truncate=False
)

+--------+-------------------+-----------------+-------------+
|line_ref|published_line_name|observation_count|vehicle_count|
+--------+-------------------+-----------------+-------------+
|6       |6                  |80               |47           |
|74      |74                 |55               |29           |
|4       |4                  |54               |34           |
|9       |9                  |52               |29           |
|7       |7                  |52               |31           |
|11A     |11A                |48               |34           |
|3       |3                  |47               |28           |
|16      |16                 |42               |24           |
|11      |11                 |41               |25           |
|11C     |11C                |41               |30           |
+--------+-------------------+-----------------+-------------+



In [34]:
# display how spark plans to process the dataframe
vehicle_clean_df.explain(
    mode="formatted"
)

== Physical Plan ==
AdaptiveSparkPlan (8)
+- == Final Plan ==
   ShuffleQueryStage (6), Statistics(sizeInBytes=750.0 KiB, rowCount=2.20E+3)
   +- Exchange (5)
      +- * Project (4)
         +- * Project (3)
            +- * Filter (2)
               +- * Scan ExistingRDD (1)
+- == Initial Plan ==
   Exchange (7)
   +- Project (4)
      +- Project (3)
         +- Filter (2)
            +- Scan ExistingRDD (1)


(1) Scan ExistingRDD [codegen id : 1]
Output [19]: [source_file#0, snapshot_time_raw#1, recorded_at_time_raw#2, valid_until_time_raw#3, operator_ref#4, line_ref#5, published_line_name#6, direction_ref#7, dated_vehicle_journey_ref#8, journey_pattern_ref#9, vehicle_ref#10, block_ref#11, origin_ref#12, destination_ref#13, destination_name#14, origin_aimed_departure_time_raw#15, longitude_raw#16, latitude_raw#17, bearing_raw#18]
Arguments: [source_file#0, snapshot_time_raw#1, recorded_at_time_raw#2, valid_until_time_raw#3, operator_ref#4, line_ref#5, published_line_name#6, direction

In [35]:
# create the processed output folder when it does not exist
processed_folder.mkdir(
    parents=True,
    exist_ok=True
)

print("processed folder:", processed_folder)

processed folder: /Users/babitaadhikari/Desktop/bus-disruption-platform/data/processed/vehicle_locations


In [36]:
# save the cleaned vehicle location dataframe as parquet
(
    vehicle_clean_df.write
    .mode("overwrite")
    .parquet(str(processed_folder))
)
print("processed vehicle locations saved")
print("saved location:", processed_folder)

[Stage 54:>                                                         (0 + 4) / 4]

processed vehicle locations saved
saved location: /Users/babitaadhikari/Desktop/bus-disruption-platform/data/processed/vehicle_locations


In [37]:
# read the saved parquet dataset back into pyspark
saved_vehicle_df = spark.read.parquet(
    str(processed_folder)
)
# count the saved rows
saved_vehicle_count = (
    saved_vehicle_df.count()
)
print(
    "original processed rows:",
    f"{processed_vehicle_count:,}"
)
print(
    "saved parquet rows:",
    f"{saved_vehicle_count:,}")
print(
    "saved parquet partitions:",
    saved_vehicle_df.rdd.getNumPartitions()
)
# stop when the saved row count is different
if saved_vehicle_count != processed_vehicle_count:
    raise ValueError(
        "saved row count does not match processed row count"
    )
print("vehicle location parquet verification successful")

original processed rows: 2,197
saved parquet rows: 2,197
saved parquet partitions: 4
vehicle location parquet verification successful


In [38]:
# find files created inside the processed folder
saved_files = sorted(
    processed_folder.glob("*")
)
print(
    "number of generated files:",
    len(saved_files)
)
for file_path in saved_files:
    print(file_path.name)

number of generated files: 10
._SUCCESS.crc
.part-00000-1f821968-abc7-4431-80a8-ce0044ab7f4d-c000.snappy.parquet.crc
.part-00001-1f821968-abc7-4431-80a8-ce0044ab7f4d-c000.snappy.parquet.crc
.part-00002-1f821968-abc7-4431-80a8-ce0044ab7f4d-c000.snappy.parquet.crc
.part-00003-1f821968-abc7-4431-80a8-ce0044ab7f4d-c000.snappy.parquet.crc
_SUCCESS
part-00000-1f821968-abc7-4431-80a8-ce0044ab7f4d-c000.snappy.parquet
part-00001-1f821968-abc7-4431-80a8-ce0044ab7f4d-c000.snappy.parquet
part-00002-1f821968-abc7-4431-80a8-ce0044ab7f4d-c000.snappy.parquet
part-00003-1f821968-abc7-4431-80a8-ce0044ab7f4d-c000.snappy.parquet


In [41]:
# display the final notebook results
print("vehicle location preprocessing summary")
print("-" * 50)

print(
    "xml files processed:",
    len(xml_files)
)

print(
    "raw observations:",
    f"{len(all_vehicle_records):,}"
)

print(
    "processed observations:",
    f"{processed_vehicle_count:,}"
)

print(
    "duplicate observations removed:",
    f"{duplicates_removed:,}"
)

print(
    "invalid coordinate records found:",
    f"{invalid_coordinate_count:,}"
)

print(
    "distinct vehicles:",
    vehicle_clean_df
    .select("vehicle_ref")
    .distinct()
    .count()
)

print(
    "distinct lines:",
    vehicle_clean_df
    .select("line_ref")
    .distinct()
    .count()
)

print(
    "distinct operators:",
    vehicle_clean_df
    .select("operator_ref")
    .distinct()
    .count()
)

print(
    "partitions:",
    vehicle_clean_df.rdd.getNumPartitions()
)

print("saved format: parquet")
print("saved location:", processed_folder)

vehicle location preprocessing summary
--------------------------------------------------
xml files processed: 2
raw observations: 2,197
processed observations: 2,197
duplicate observations removed: 333
invalid coordinate records found: 0
distinct vehicles: 1147
distinct lines: 111
distinct operators: 2
partitions: 4
saved format: parquet
saved location: /Users/babitaadhikari/Desktop/bus-disruption-platform/data/processed/vehicle_locations


In [40]:
# count rows before duplicate removal
before_duplicate_removal = vehicle_clean_df.count()

# remove repeated observations
vehicle_clean_df = (
    vehicle_clean_df
    .dropDuplicates([
        "vehicle_ref",
        "line_ref",
        "recorded_at_time"
    ])
)

# count rows after duplicate removal
after_duplicate_removal = vehicle_clean_df.count()

# calculate the number of duplicate records removed
duplicates_removed = (
    before_duplicate_removal
    - after_duplicate_removal
)

print(
    "rows before duplicate removal:",
    f"{before_duplicate_removal:,}"
)

print(
    "rows after duplicate removal:",
    f"{after_duplicate_removal:,}"
)

print(
    "duplicate observations removed:",
    f"{duplicates_removed:,}"
)

rows before duplicate removal: 2,197
rows after duplicate removal: 1,864
duplicate observations removed: 333
